# Customer & Sales Analytics — Olist E-Commerce

Analisi end-to-end di ~100k ordini e-commerce brasiliani (2017–2018).

Fasi: EDA → RFM Segmentation → Cohort Analysis → Delivery Analysis

## 0. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# Scarica il dataset da Kaggle (https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce)
# ed estrai i 9 CSV in una cartella "dataset/" allo stesso livello di "notebooks/"
path = '../dataset/'

orders       = pd.read_csv(path + 'olist_orders_dataset.csv')
order_items  = pd.read_csv(path + 'olist_order_items_dataset.csv')
customers    = pd.read_csv(path + 'olist_customers_dataset.csv')
products     = pd.read_csv(path + 'olist_products_dataset.csv')
sellers      = pd.read_csv(path + 'olist_sellers_dataset.csv')
payments     = pd.read_csv(path + 'olist_order_payments_dataset.csv')
reviews      = pd.read_csv(path + 'olist_order_reviews_dataset.csv')
geolocation  = pd.read_csv(path + 'olist_geolocation_dataset.csv')
category_tr  = pd.read_csv(path + 'product_category_name_translation.csv')

print('Dati caricati!')

## 1. Ispezione Dataset

In [ ]:
# Dimensioni di ogni tabella
tabelle = {
    'orders': orders, 'order_items': order_items, 'customers': customers,
    'products': products, 'sellers': sellers, 'payments': payments,
    'reviews': reviews, 'geolocation': geolocation, 'category_translation': category_tr
}

for nome, df in tabelle.items():
    print(f'{nome:25} -> {df.shape[0]:>7} righe  |  {df.shape[1]} colonne')

In [ ]:
# Prime righe e struttura della tabella principale
orders.head(5)

In [ ]:
# Distribuzione degli stati degli ordini
orders['order_status'].value_counts()

In [ ]:
# Conversione colonne date da object a datetime
date_cols = [
    'order_purchase_timestamp', 'order_approved_at',
    'order_delivered_carrier_date', 'order_delivered_customer_date',
    'order_estimated_delivery_date'
]
for col in date_cols:
    orders[col] = pd.to_datetime(orders[col])

print(orders.dtypes)

## 2. Data Quality Check

In [ ]:
# Valori null in orders
# I null su date di consegna sono attesi per ordini cancellati
orders.isnull().sum()

In [ ]:
# Range temporale del dataset
print('Data minima:', orders['order_purchase_timestamp'].min())
print('Data massima:', orders['order_purchase_timestamp'].max())

In [ ]:
# Totale valori null per ogni tabella
# reviews ha molti null: i campi testo della recensione sono opzionali
for nome, df in tabelle.items():
    print(f'{nome:25} -> {df.isnull().sum().sum()} null totali')

In [ ]:
# Controllo duplicati sulla chiave primaria di orders
print('order_id duplicati:', orders['order_id'].duplicated().sum())

In [ ]:
# Anomalie temporali: verifica che ogni fase della filiera avvenga dopo la precedente
# Sequenza attesa: acquisto -> approvazione -> corriere -> consegna
lista_di_coppie = (
    ('order_purchase_timestamp',    'order_approved_at'),
    ('order_approved_at',           'order_delivered_carrier_date'),
    ('order_delivered_carrier_date','order_delivered_customer_date'),
    ('order_purchase_timestamp',    'order_estimated_delivery_date')
)

for col1, col2 in lista_di_coppie:
    subset = orders[~orders[col1].isnull() & ~orders[col2].isnull()].copy()
    subset['diff'] = subset[col2] - subset[col1]
    anomalie = (subset['diff'] < pd.Timedelta(0)).sum()
    print(f'{col1} -> {col2}: {anomalie} anomalie')

In [ ]:
# Anomalie numeriche: prezzi e pagamenti non devono essere negativi o zero
print('Prezzi <= 0 in order_items:', (order_items['price'] <= 0).sum())
print('Pagamenti <= 0 in payments:', (payments['payment_value'] <= 0).sum())

## 3. Analisi Esplorativa (EDA)

In [ ]:
# Aggiunta colonna mese e filtro mesi incompleti (set/ott 2016, set/ott 2018)
orders['order_month'] = orders['order_purchase_timestamp'].dt.to_period('M')

orders_clean = orders.copy()
orders_clean = orders_clean[
    (orders_clean['order_month'] >= '2017-01') &
    (orders_clean['order_month'] <= '2018-08')
]

In [ ]:
# Trend mensile ordini — picco novembre 2017 = Black Friday
monthly_orders = orders_clean.groupby('order_month')['order_id'].count()

monthly_orders.plot()
plt.title('Numero di ordini per mese')
plt.xlabel('Mese')
plt.ylabel('Numero di ordini')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Distribuzione ordini per stato brasiliano — SP (São Paulo) domina
customer_by_state = customers.groupby('customer_state')['customer_id'].count().sort_values(ascending=False)

customer_by_state.plot(kind='bar')
plt.title('Numero di clienti per stato')
plt.xlabel('Stato')
plt.ylabel('Numero di clienti')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Revenue per ordine: join orders x order_items, somma price + freight per order_id
orders_valid = orders_clean.merge(order_items, on='order_id', how='inner')
orders_valid['total_value'] = orders_valid['price'] + orders_valid['freight_value']

order_revenue = orders_valid.groupby('order_id')['total_value'].sum()
order_revenue.describe()

In [ ]:
# Distribuzione revenue — coda lunga verso destra (outlier oltre 1000)
order_revenue.plot(kind='hist', bins=50)
plt.xlim(0, 1000)
plt.title('Distribuzione revenue per ordine')
plt.xlabel('Valore ordine (BRL)')
plt.ylabel('Frequenza')
plt.tight_layout()
plt.show()

## 4. RFM Segmentation

R = Recency (giorni dall'ultimo acquisto), F = Frequency (numero ordini), M = Monetary (spesa totale).
Ogni metrica viene scorata da 1 a 5 e combinata in un punteggio RFM.

In [ ]:
# Join con customers per portare customer_unique_id (ID persona reale, non per-ordine)
rfm_table = orders_valid.merge(customers, on='customer_id', how='inner')

In [ ]:
# Calcolo delle tre metriche RFM per customer_unique_id
frequency = rfm_table.groupby('customer_unique_id')['order_id'].count()
monetary  = rfm_table.groupby('customer_unique_id')['total_value'].sum()

# Recency: giorni dall'ultimo acquisto rispetto alla data massima del dataset
recency = (
    rfm_table['order_purchase_timestamp'].max() -
    rfm_table.groupby('customer_unique_id')['order_purchase_timestamp'].max()
).dt.days

rfm = pd.DataFrame({'recency': recency, 'frequency': frequency, 'monetary': monetary})
rfm.head()

In [ ]:
# Scoring 1-5 per ogni metrica
# R: invertito (recency bassa = cliente recente = score alto)
# F: bin manuali perché il 90% dei clienti ha F=1 (qcut non funzionerebbe)
rfm['R'] = pd.qcut(rfm['recency'], q=5, labels=[5,4,3,2,1])
rfm['F'] = pd.cut(rfm['frequency'], bins=[0,1,2,3,5,25], labels=[1,2,3,4,5])
rfm['M'] = pd.qcut(rfm['monetary'], q=5, labels=[1,2,3,4,5])

rfm['RFM_score'] = rfm['R'].astype(str) + rfm['F'].astype(str) + rfm['M'].astype(str)
rfm.head()

In [ ]:
# Assegnazione segmenti di business tramite np.select
# L'ordine delle condizioni è importante: la prima vera vince
condizioni = [
    (rfm['R'].astype(int) == 5) & (rfm['F'].astype(int) == 5),       # Champion
    rfm['F'].astype(int) >= 4,                                         # Loyal
    (rfm['R'].astype(int) <= 2) & (rfm['F'].astype(int) >= 3),        # At Risk
    (rfm['R'].astype(int) == 1) & (rfm['F'].astype(int) == 1),        # Lost
    (rfm['R'].astype(int) == 5) & (rfm['F'].astype(int) == 1),        # New Customer
    (rfm['R'].astype(int) >= 4) & (rfm['F'].astype(int) == 1),        # Promising
    rfm['R'].astype(int).isin([2,3]) & (rfm['F'].astype(int) == 1),   # Hibernating
]
segmenti = ['Champion', 'Loyal', 'At Risk', 'Lost', 'New Customer', 'Promising', 'Hibernating']

rfm['segment'] = np.select(condizioni, segmenti, default='Other')
rfm['segment'].value_counts()

In [ ]:
rfm['segment'].value_counts().plot(kind='bar')
plt.title('Distribuzione segmenti RFM')
plt.xlabel('Segmento')
plt.ylabel('Numero clienti')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 5. Cohort Analysis

Per ogni coorte (mese del primo acquisto) si misura quanti clienti tornano nei mesi successivi.
Il risultato atteso è una retention molto bassa (~1%), tipica dell'e-commerce con acquisto unico.

In [ ]:
# Mese del primo acquisto per cliente (coorte di appartenenza)
first_purchase = rfm_table.groupby('customer_unique_id')['order_purchase_timestamp'].min().dt.to_period('M')

rfm_table['cohort']       = rfm_table['customer_unique_id'].map(first_purchase)
rfm_table['cohort_index'] = (rfm_table['order_month'] - rfm_table['cohort']).apply(lambda x: x.n)

In [ ]:
# Tabella pivot: righe = coorte, colonne = mesi dall'acquisto, valori = clienti unici attivi
cohort_table = rfm_table.groupby(['cohort', 'cohort_index'])['customer_unique_id'].nunique().unstack()

# Conversione in percentuale rispetto alla dimensione iniziale della coorte (colonna 0)
cohort_pct = cohort_table.divide(cohort_table[0], axis=0)

In [ ]:
# Heatmap retention: quasi tutto rosso (0%) conferma il basso tasso di riacquisto
plt.figure(figsize=(14, 7))
sns.heatmap(cohort_pct, annot=True, fmt='.0%', cmap='YlOrRd_r')
plt.title('Cohort Retention Rate')
plt.xlabel('Mesi dal primo acquisto')
plt.ylabel('Coorte (mese primo acquisto)')
plt.tight_layout()
plt.show()

## 6. Delivery Analysis

Analisi dei tempi di consegna e impatto dei ritardi sul review score del cliente.

In [ ]:
# Filtro: solo ordini consegnati con data di consegna presente
delivered_orders = orders_clean[
    (orders_clean['order_status'] == 'delivered') &
    (~orders_clean['order_delivered_customer_date'].isnull())
].copy()

# Giorni di consegna (acquisto -> consegna al cliente)
delivered_orders['delivery_days'] = (
    delivered_orders['order_delivered_customer_date'] -
    delivered_orders['order_purchase_timestamp']
).dt.days

# Flag ritardo: consegnato dopo la data stimata
delivered_orders['is_late'] = (
    delivered_orders['order_delivered_customer_date'] >
    delivered_orders['order_estimated_delivery_date']
)

print('Statistiche giorni di consegna:')
print(delivered_orders['delivery_days'].describe())
print()
print('Ordini in ritardo:', delivered_orders['is_late'].sum())
print(f'% in ritardo: {delivered_orders["is_late"].mean():.1%}')

In [ ]:
# Correlazione ritardo e review score: join con tabella reviews
delivery_reviews = delivered_orders.merge(reviews, on='order_id', how='inner')

review_score_per_late = delivery_reviews.groupby('is_late')['review_score'].mean()
print(review_score_per_late)

In [ ]:
# Review score medio: 4.3 per ordini puntuali vs 2.6 per ordini in ritardo
review_score_per_late.plot(kind='bar')
plt.title('Review Score medio: puntuale vs in ritardo')
plt.xlabel('In ritardo')
plt.ylabel('Review Score medio')
plt.xticks(ticks=[0,1], labels=['Puntuale', 'In ritardo'], rotation=0)
plt.ylim(0, 5)
plt.tight_layout()
plt.show()